In [3]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.io.sink import RingBuffer as Sink
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

# Create a single LIF neuron
# vth = threshold, dv = voltage decay, du = current decay
lif_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=10) # Changing bias_mant=5 (33 spikes) to bias_mant=10 (50 spikes)

# Create a sink to record the neuron's spike output
sink = Sink(shape=(1,), buffer=100)

# Connect the neuron's spike output to the sink
lif_neuron.s_out.connect(sink.a_in)

# Run for 100 timesteps
lif_neuron.run(condition=RunSteps(num_steps=100), 
               run_cfg=Loihi2SimCfg())

# Get the recorded spike data
spike_data = sink.data.get()

lif_neuron.stop()

print(f"Spike data shape: {spike_data.shape}")
print(f"Total spikes: {np.sum(spike_data)}")
print(f"Spike train: {spike_data.flatten()}")

Spike data shape: (1, 100)
Total spikes: 50.0
Spike train: [0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1.]


In [5]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.io.sink import RingBuffer as Sink
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

lif_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)

# This time, record voltage directly using Lava's monitor
from lava.proc.monitor.process import Monitor

monitor = Monitor()
monitor.probe(lif_neuron.v, 100)  # probe the voltage variable for 100 steps

lif_neuron.run(condition=RunSteps(num_steps=100), 
               run_cfg=Loihi2SimCfg())

voltage_data = monitor.get_data()
lif_neuron.stop()

v_trace = voltage_data['LIF']['v'].flatten()
print("Voltage trace (first 15 steps):")
print(v_trace[:15])

KeyError: 'LIF'

In [6]:
print(voltage_data.keys())

dict_keys(['Process_8'])


In [7]:
v_trace = voltage_data['Process_8']['v'].flatten()
print("Voltage trace (first 15 steps):")
print(v_trace[:15])

Voltage trace (first 15 steps):
[ 5. 10.  0.  5. 10.  0.  5. 10.  0.  5. 10.  0.  5. 10.  0.]


## Lava LIF Neuron — Observations & Results

### What is Lava?
Intel's open-source neuromorphic computing framework, designed specifically 
for Loihi hardware. Unlike Brian2's continuous differential equation approach, 
Lava operates on **discrete timesteps**, mirroring how real neuromorphic 
chips process information in clock cycles.

### Architecture
A single LIF neuron process with:
- `vth` — firing threshold
- `dv` — voltage decay rate (0 = no decay)
- `du` — current decay rate (0 = no decay)  
- `bias_mant` — constant current injected every timestep

### Key finding 1: Bias controls firing rate
| `bias_mant` | Total spikes (100 steps) | Pattern |
|-------------|--------------------------|---------|
| 5 | 33 | `0, 0, 1` repeating (fires every 3rd step) |
| 10 | 50 | `0, 1` repeating (fires every 2nd step) |

With `vth=10`, a bias of 5 requires 2 accumulation steps to cross threshold 
(fires on the 3rd). A bias of 10 crosses threshold every single step in 
principle, but actually fires every other step due to Lava's internal timing.

### Key finding 2: One-step delay between threshold crossing and spike reporting
Initially expected `bias_mant=10` to produce 100/100 spikes (firing every 
step), but observed an alternating `0, 1` pattern instead (50 spikes). This 
suggested some kind of timing delay in Lava's discrete-time simulation, so 
the membrane voltage was probed directly using Lava's `Monitor` process to 
investigate further.

**Voltage trace (bias_mant=5, vth=10):**
Step 0:  v=5   (bias added, below threshold, no spike)

Step 1:  v=10  (bias added again, voltage reaches threshold)

Step 2:  v=0   (spike reported, neuron resets)

Step 3:  v=5   (cycle repeats)

**Conclusion:** Voltage crosses threshold at timestep N, but the spike is 
only reported and the reset only occurs at timestep N+1. This is a one-step 
internal delay between threshold crossing and spike reporting in Lava's 
discrete-time simulation.

This directly explains the earlier 33-spike result with `bias_mant=5` — the 
`0, 0, 1` pattern corresponds to: voltage building (step 0), threshold reached 
(step 1), spike reported and reset (step 2).

**Significance:** When interpreting any Lava spike train, the spike recorded 
at timestep N actually corresponds to threshold crossing at timestep N-1. 
This is a genuine, reproducible feature of Lava's internal processing — 
discovered here through direct voltage probing rather than assumed — and is 
directly relevant to understanding real Loihi 2 hardware timing.

**Note on debugging:** The `Monitor` process names probed variables based on an auto-incrementing 
process counter (e.g. `Process_8`) rather than the class name (`LIF`). This 
counter can change between notebook runs, so checking `voltage_data.keys()` 
before accessing data is good practice.